# 1. Rank: counting independent information

After [inverse and pseudoinverse](12_matrix_inverse_and_pseudoinverse.ipynb), you know some systems lose information. Rank counts how many independent directions remain. You will recognize redundant columns, perform exact elimination on small integer matrices, compare exact and numerical rank, and explain why multiple coefficient vectors can make identical predictions. This is an algebra lesson, not a feature-selection recipe or a fitted predictive model.

## 2. What problem does this solve?

Suppose a table stores distance in meters and the same distance in centimeters. Two columns are present, but knowing the first determines the second exactly. A model cannot identify two separate effects from these duplicated measurements. Rank helps distinguish the number of stored columns from the amount of independent linear information they contain. It also tells us when a square transformation is reversible and when a system has redundant equations.

## 3. Intuition and analogy

Think of walking on a map. A step east and twice that step east offer only one direction. East and north allow movement across a plane. Combining three different eastward steps still does not create a northward direction. Independence means that one direction cannot be built by scaling and adding the others. The map analogy is about possible movement, not about which directions are statistically useful for predicting a target.

## 4. Terminology

A linear combination multiplies each vector by a scalar and adds the results. The span is the collection of all such combinations. Independent columns contribute new directions; dependent columns contain at least one redundant direction. Rank is the dimension of their span. Row rank counts independent rows and equals column rank, although their spans live in different coordinate spaces. A pivot is a nonzero leading entry used to eliminate other entries during row reduction. Numerical rank treats sufficiently tiny directions as zero at a specified tolerance.

## 5. Mathematical foundations

Let \(X\) have shape \((m,p)\), meaning \(m\) rows and \(p\) columns. Its columns \(v_1,\ldots,v_p\) each have length \(m\). They are independent precisely when
\[
c_1v_1+\cdots+c_pv_p=0\quad\Rightarrow\quad c_1=\cdots=c_p=0.
\]
Each \(c_j\) is a scalar multiplier; zero on the left is a length-\(m\) zero vector. If a nonzero choice of multipliers still produces zero, the columns are dependent. The largest independent collection has size \(r=\operatorname{rank}(X)\), with
\[
0\le r\le\min(m,p).
\]
Here \(r\), \(m\), and \(p\) are counts, not measured physical quantities. Full column rank means \(r=p\). A square \(p\)-by-\(p\) matrix is invertible exactly when its rank equals \(p\). This bridges rank and the preceding lesson's failure of inversion.

## 6. Hand-calculated example

Take \(X=[[1,2],[2,4],[3,6]]\). Column two is twice column one. Multipliers \(c_1=-2,c_2=1\) produce the zero vector, so the two columns are dependent. Column one is nonzero, so rank is exactly one, rather than zero.

For row reduction, subtract twice the first row from the second: [2,4]-2[1,2]=[0,0]. Subtract three times the first from the third: [3,6]-3[1,2]=[0,0]. One nonzero pivot row remains. In contrast, rows [1,0], [2,1], [3,0] reduce to pivots in both columns. Their matrix has rank two even though it still has only three observations.

## 7. Scratch implementation: exact elimination

`Fraction` represents rational numbers as integer numerator and denominator, avoiding floating-point rounding for our small integer inputs. This function copies its input into a list of rows. A pivot row is swapped into place, normalized, and subtracted from rows below. Python's `next` finds the first qualifying row, with `None` indicating no pivot. This slow teaching implementation supports rectangular integer tables only; it is not a replacement for numerical linear algebra on measured data.

In [ ]:
# Enable embedded figures in this fresh notebook kernel.
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

In [ ]:
from fractions import Fraction
import numpy as np

def exact_integer_rank(values):
    original = np.asarray(values)
    if original.ndim != 2 or 0 in original.shape:
        raise ValueError('Use a nonempty rectangular matrix.')
    if original.dtype.kind not in 'iu':
        raise ValueError('This exact teaching function accepts integers only.')
    rows = [[Fraction(int(v)) for v in row] for row in original]
    pivot_count = 0
    for column in range(original.shape[1]):
        pivot = next((i for i in range(pivot_count, len(rows))
                      if rows[i][column] != 0), None)
        if pivot is None:
            continue
        rows[pivot_count], rows[pivot] = rows[pivot], rows[pivot_count]
        divisor = rows[pivot_count][column]
        rows[pivot_count] = [v/divisor for v in rows[pivot_count]]
        for i in range(pivot_count + 1, len(rows)):
            factor = rows[i][column]
            rows[i] = [v - factor*w for v, w in zip(rows[i], rows[pivot_count])]
        pivot_count += 1
        if pivot_count == len(rows):
            break
    return pivot_count, rows

X = np.array([[1, 2], [2, 4], [3, 6]])
rank, reduced = exact_integer_rank(X)
assert rank == 1
assert reduced == [[Fraction(1), Fraction(2)], [0, 0], [0, 0]]
print('Exact rank:', rank)

## 8. Inspecting intermediate values

The returned rows are in echelon form: every later pivot would occur farther right. Zero rows contribute no independent equation. Our procedure stops after finding as many pivots as rows; it does not need to eliminate above pivots to count rank. Keeping the original unchanged matters when later cells compare original measurements with transformed equations.

In [ ]:
for i, row in enumerate(reduced):
    print('Reduced row', i, ':', [str(value) for value in row])
independent = np.array([[1, 0], [2, 1], [3, 0]])
assert exact_integer_rank(independent)[0] == 2
assert exact_integer_rank(np.zeros((3, 2), dtype=int))[0] == 0
assert exact_integer_rank([[0, 1], [1, 0]])[0] == 2  # Requires swapping.
assert np.array_equal(X, [[1, 2], [2, 4], [3, 6]])
print('Independent example rank:', exact_integer_rank(independent)[0])

## 9. Library implementation

NumPy's `matrix_rank` estimates rank using singular values, strengths of independent directions studied in a later lesson. `tol` sets an absolute cutoff for treating those strengths as zero. Exact dependence and practical indistinguishability are different questions. Tiny measured differences might be meaningful or might be noise; the computer cannot decide their scientific meaning for you.

In [ ]:
assert np.linalg.matrix_rank(X) == 1
assert np.linalg.matrix_rank(independent) == 2
almost_flat = np.diag([1., 1e-8])
assert np.linalg.matrix_rank(almost_flat, tol=1e-10) == 2
assert np.linalg.matrix_rank(almost_flat, tol=1e-6) == 1
print('Numerical ranks at two tolerances:',
      np.linalg.matrix_rank(almost_flat, tol=1e-10),
      np.linalg.matrix_rank(almost_flat, tol=1e-6))

## 10. Visualization

Each point below is a row with two feature coordinates. On the left all observations lie on a line through the origin: their row span has one direction. On the right at least two rows point in independent directions, so their row span is the plane. Rank concerns linear span through the origin. Points on a line offset from the origin can have rank two before centering, a subtle distinction explored below.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in zip(axes, [X, independent], ['Rank 1', 'Rank 2']):
    for row in data:
        ax.plot([0, row[0]], [0, row[1]], alpha=.55)
    ax.scatter(data[:, 0], data[:, 1], s=70)
    ax.set(xlabel='Feature 1', ylabel='Feature 2', title=title,
           xlim=(-.5, 3.5), ylim=(-.5, 6.5))
    ax.set_aspect('equal')
    ax.grid(alpha=.2)
display(fig)
plt.close(fig)

## 11. A synthetic feature-table experiment

If features are exact duplicates up to scale, individual weights cannot be uniquely identified. Multiplying our table by weights [3,0] gives the same outputs as [1,1], since the second column is twice the first. This does not mean predictions are necessarily bad. It means a claim about the separate effect of each redundant column is unsupported. An intercept column can also introduce dependence if another column is constant.

In [ ]:
weights_a = np.array([3., 0.])
weights_b = np.array([1., 1.])
assert np.allclose(X @ weights_a, X @ weights_b)
print('Different weights, identical outputs:', X @ weights_a, X @ weights_b)
offset_line = np.array([[1, 3], [2, 4], [3, 5]])
centered = offset_line - offset_line.mean(axis=0)
assert np.linalg.matrix_rank(offset_line) == 2
assert np.linalg.matrix_rank(centered) == 1
print('Rank before and after centering:',
      np.linalg.matrix_rank(offset_line), np.linalg.matrix_rank(centered))

## 12. Configuration rather than training

There are no learned model parameters in a rank calculation. The exact algorithm accepts an integer matrix and counts pivots. The numerical algorithm accepts a tolerance, a computational choice. Units and feature scaling affect absolute cutoffs. Centering changes the matrix being analyzed: it answers a question about variation around a mean, rather than the original position vectors. In a predictive workflow, means and feature-removal rules must be chosen using training data only.

## 13. Evaluation and interpretation

Test known ranks, swapped pivots, zero matrices, and rectangular inputs. For numerical data, report a tolerance or explain reliance on the library default. Investigate whether conclusions survive plausible measurement changes. Full rank does not guarantee good conditioning: the almost-flat example is exactly invertible yet sensitive. Rank also does not measure predictive value. An independent column of random noise may increase rank without helping a model generalize.

## 14. Assumptions and limitations

Rank describes linear relationships. A column containing the square of another can be linearly independent even though there is a deterministic nonlinear relationship. Exact rational arithmetic grows expensive as numerators and denominators grow. Floating-point rank reflects the precision and threshold used; it cannot establish exact mathematical dependence from noisy observations. Missing values must be handled deliberately before applying a numerical rank routine.

## 15. Common mistakes and debugging

Counting nonzero columns overestimates rank when columns repeat. Counting unique rows also fails because distinct rows can be multiples. Confusing matrix shape with rank overlooks redundancy. Treating a tolerance-induced rank change as a discovered physical law mistakes numerical policy for evidence. When results surprise you, inspect units, centering, duplicated columns, and whether an intercept was included.

## 16. Comparison with alternatives

Correlation summarizes pairwise association, whereas rank detects dependence involving several columns together. Three columns can satisfy column three equals columns one plus two without any pair being exact multiples. Singular values provide a graded view of directional strengths rather than one integer. Feature selection additionally considers a task, validation design, and costs. None of these quantities alone proves that a feature causes an outcome.

## 17. Exercises

1. **Beginner:** Determine the rank of [[0,0],[0,0]] and explain why.
2. **Beginner:** Determine the rank of [[1,2],[2,4]] without software.
3. **Beginner:** Construct a three-by-two matrix of rank two.
4. **Intermediate:** Add a column equal to the sum of two independent columns. Explain its rank.
5. **Intermediate:** Explain the change in rank caused by centering the offset-line example.
6. **Challenge:** Construct a nonzero weight change that leaves predictions unchanged for a table whose third column equals its first plus its second.

## 18. Worked solutions

1. Rank is zero because the span contains only the zero vector, with no independent direction.
2. The second column equals twice the first and the first is nonzero, so rank is one.
3. Rows [1,0], [0,1], [1,1] work: the first two already provide independent directions, and rank cannot exceed two columns.
4. Rank stays two. The added column contributes no new direction, even though it may differ from every existing column.
5. Before centering, two position vectors on y=x+2 point in different directions from the origin. Subtracting the mean moves the line through the origin, leaving one direction of variation.
6. Change the three weights by [1,1,-1]. The prediction change is column one plus column two minus column three, exactly zero. Any scalar multiple of that change also preserves predictions. This gives infinitely many indistinguishable coefficient vectors.

In [ ]:
base = np.array([[1, 0], [0, 1], [1, 1]])
extended = np.column_stack([base, base.sum(axis=1)])
assert exact_integer_rank(base)[0] == 2
assert exact_integer_rank(extended)[0] == 2
assert np.array_equal(extended @ [1, 1, -1], [0, 0, 0])
try:
    exact_integer_rank([[1.5, 2.]])
except ValueError:
    print('Noninteger input correctly rejected by exact teaching routine.')
else:
    raise AssertionError('Integer-only contract was not enforced.')

## 19. Knowledge check

**Can rank exceed the number of rows?** No; vectors of that length have at most that many independent coordinates.

**Can two nonzero columns have rank one?** Yes, when one is a scalar multiple of the other.

**Is full rank the same as stable inversion?** No; a full-rank matrix may be nearly singular.

**Does centering preserve rank?** Not necessarily; subtracting column means changes position vectors.

**Does rank detect every kind of redundancy?** No; it describes linear dependence, not all nonlinear relationships.

## 20. Interview preparation

**Why can regression coefficients be nonunique?** A nonzero weight change in the matrix's null space leaves all fitted outputs unchanged.

**What does numerical rank add to exact rank?** It treats directions below a tolerance as indistinguishable from zero at the chosen scale.

**Why can pairwise checks miss dependence?** A column can be a combination of several others without duplicating any single one.

**How are rank and invertibility related?** A square matrix is invertible exactly when it has full rank.

**Should rank alone select model features?** No; relevance, validation, measurement noise, and deployment needs also matter.

## 21. Summary and next steps

Rank counts independent linear directions, not stored values or predictive usefulness. Exact dependence and practical numerical dependence require different interpretations. Continue to [norms and distances](14_norms_and_distances.ipynb) to quantify vector length and separation.